# Signal Generation Demo

This notebook demonstrates the signal generation capabilities of the Neural Receiver system.

We'll explore:
1. Generating various modulation types
2. Visualizing IQ data in different domains
3. Understanding the dataset structure

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from src.data import SignalGenerator, SignalParams, ModulationType
from src.utils import (
    plot_iq_data, plot_spectrogram, plot_psd, 
    plot_constellation, plot_signal_examples
)

## 1. Initialize Signal Generator

In [ ]:
# Create signal generator
generator = SignalGenerator(sample_rate=1.0, seed=42)

# Get available modulation types
class_names = ModulationType.get_class_names()
print(f"Available modulation types ({len(class_names)}):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

## 2. Generate Example Signals

Let's generate examples of each modulation type and visualize them.

In [ ]:
# Generate AM signal
am_params = SignalParams(
    center_freq=0.1,
    bandwidth=0.05,
    power=1.0,
    snr_db=-5.0,
    symbol_rate=200,
    modulation_type='AM'
)

signal_am, noisy_am = generator.generate_signal(1024, am_params)
print(f"Generated AM signal: {len(signal_am)} samples, SNR = {am_params.snr_db} dB")

# Visualize
plot_iq_data(noisy_am, title="AM Modulated Signal (SNR = -5 dB)")

In [ ]:
# Generate FM signal
fm_params = SignalParams(
    center_freq=0.0,
    bandwidth=0.1,
    power=1.0,
    snr_db=-3.0,
    symbol_rate=150,
    modulation_type='FM'
)

signal_fm, noisy_fm = generator.generate_signal(1024, fm_params)
plot_iq_data(noisy_fm, title="FM Modulated Signal (SNR = -3 dB)")

In [ ]:
# Generate LFM Chirp (Radar)
chirp_params = SignalParams(
    center_freq=0.0,
    bandwidth=0.15,
    power=1.0,
    snr_db=-8.0,
    symbol_rate=100,
    modulation_type='LFM/Chirp'
)

signal_chirp, noisy_chirp = generator.generate_signal(1024, chirp_params)
plot_iq_data(noisy_chirp, title="LFM Chirp Signal (SNR = -8 dB)")

## 3. Spectral Analysis

In [ ]:
# Plot spectrogram for chirp signal
plot_spectrogram(noisy_chirp, nperseg=128, title="LFM Chirp Spectrogram")

In [ ]:
# Plot PSD
plot_psd(noisy_am, nperseg=512, title="AM Signal - Power Spectral Density")

## 4. Constellation Diagrams

In [ ]:
# 2FSK signal
fsk_params = SignalParams(
    center_freq=0.05,
    bandwidth=0.08,
    power=1.0,
    snr_db=0.0,
    symbol_rate=250,
    modulation_type='2FSK'
)

signal_fsk, noisy_fsk = generator.generate_signal(2048, fsk_params)
plot_constellation(noisy_fsk, title="2FSK Constellation (SNR = 0 dB)")

## 5. Dataset Exploration

In [ ]:
from src.data import SignalDataset

# Create a small dataset
dataset = SignalDataset(
    n_samples=100,
    sequence_length=1024,
    snr_range=(-10, 0),
    no_signal_prob=0.2,
    seed=42
)

print(f"Dataset size: {len(dataset)}")

# Get a sample
iq_tensor, labels = dataset[0]
print(f"\nSample shape: {iq_tensor.shape}")
print(f"Labels: {labels.keys()}")
print(f"  Signal present: {labels['signal_present'].item()}")
print(f"  Modulation class: {labels['modulation_class'].item()} ({class_names[labels['modulation_class'].item()]})")
print(f"  SNR: {labels['snr_db'].item():.2f} dB")

In [ ]:
# Visualize random examples from dataset
plot_signal_examples(dataset, n_examples=4)

## 6. SNR Impact Analysis

In [ ]:
# Generate same signal at different SNR levels
snr_levels = [-10, -5, 0, 5]
fig, axes = plt.subplots(len(snr_levels), 2, figsize=(14, 3*len(snr_levels)))

for i, snr in enumerate(snr_levels):
    params = SignalParams(
        center_freq=0.1,
        bandwidth=0.08,
        power=1.0,
        snr_db=snr,
        symbol_rate=200,
        modulation_type='AM'
    )
    
    signal, noisy = generator.generate_signal(1024, params)
    
    # Time domain
    time = np.arange(len(noisy))
    axes[i, 0].plot(time, noisy.real, 'b-', alpha=0.7)
    axes[i, 0].plot(time, noisy.imag, 'r-', alpha=0.7)
    axes[i, 0].set_title(f'Time Domain (SNR = {snr} dB)')
    axes[i, 0].set_ylabel('Amplitude')
    axes[i, 0].grid(True, alpha=0.3)
    
    # Frequency domain
    fft = np.fft.fftshift(np.fft.fft(noisy))
    freq = np.fft.fftshift(np.fft.fftfreq(len(noisy)))
    mag_db = 20 * np.log10(np.abs(fft) + 1e-10)
    
    axes[i, 1].plot(freq, mag_db, 'g-')
    axes[i, 1].set_title(f'Spectrum (SNR = {snr} dB)')
    axes[i, 1].set_ylabel('Magnitude (dB)')
    axes[i, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Sample')
axes[-1, 1].set_xlabel('Normalized Frequency')
plt.tight_layout()
plt.show()

## Summary

In this notebook, we demonstrated:
- Generation of various modulation types (AM, FM, 2FSK, radar waveforms)
- Visualization in time, frequency, and constellation domains
- Dataset structure for multi-task learning
- Impact of SNR on signal characteristics

Next steps:
- Train the neural receiver model (see `02_training_demo.ipynb`)
- Evaluate performance (see `03_evaluation_demo.ipynb`)